In [2]:
%pip install pandas plotly geopy nbformat
import pandas as pd 

Note: you may need to restart the kernel to use updated packages.


In [3]:
df = pd.read_csv ('/home/ciline/Téléchargements/ship_fuel_efficiency.csv')

In [4]:
df.head(20)

,ship_id,ship_type,route_id,month,distance,fuel_type,fuel_consumption,CO2_emissions,weather_conditions,engine_efficiency
0,NG001,Oil Service Boat,Warri-Bonny,January,132.26,HFO,3779.77,10625.76,Stormy,92.14
1,NG001,Oil Service Boat,Port Harcourt-Lagos,February,128.52,HFO,4461.44,12779.73,Moderate,92.98
2,NG001,Oil Service Boat,Port Harcourt-Lagos,March,67.30,HFO,1867.73,5353.01,Calm,87.61
3,NG001,Oil Service Boat,Port Harcourt-Lagos,April,71.68,Diesel,2393.51,6506.52,Stormy,87.42
4,NG001,Oil Service Boat,Lagos-Apapa,May,134.32,HFO,4267.19,11617.03,Calm,85.61
5,NG001,Oil Service Boat,Port Harcourt-Lagos,June,85.93,Diesel,2342.13,6753.42,Stormy,72.82
6,NG001,Oil Service Boat,Warri-Bonny,July,85.67,HFO,2974.79,8498.16,Moderate,93.93
7,NG001,Oil Service Boat,Warri-Bonny,August,44.81,Diesel,1376.38,3509.56,Moderate,91.10
8,NG001,Oil Service Boat,Escravos-Lagos,September,116.44,Diesel,3661.75,9423.97,Calm,73.41
9,NG001,Oil Service Boat,Lagos-Apapa,October,99.73,HFO,2551.99,6416.66,Moderate,94.68


In [5]:
print(df.columns)

Index(['ship_id', 'ship_type', 'route_id', 'month', 'distance', 'fuel_type',
       'fuel_consumption', 'CO2_emissions', 'weather_conditions',
       'engine_efficiency'],
      dtype='str')


In [6]:
existingSHIPS = df.iloc [:,0].unique()
print (f' there are {len(df)} lines in the file \n {len(existingSHIPS)} existing ships in the dataset identified as follows ')
print (existingSHIPS)
#the firs column represents the ships names , 

 there are 1440 lines in the file 
 120 existing ships in the dataset identified as follows 
<StringArray>
['NG001', 'NG002', 'NG003', 'NG004', 'NG005', 'NG006', 'NG007', 'NG008',
 'NG009', 'NG010',
 ...
 'NG111', 'NG112', 'NG113', 'NG114', 'NG115', 'NG116', 'NG117', 'NG118',
 'NG119', 'NG120']
Length: 120, dtype: str


In [7]:
existingSHIPtype  = df.loc [:,'ship_type'].unique()
print (existingSHIPtype)

<StringArray>
['Oil Service Boat', 'Fishing Trawler', 'Surfer Boat', 'Tanker Ship']
Length: 4, dtype: str


In [8]:
existingFUEL = df.loc[:,'fuel_type'].unique()
print(existingFUEL)

<StringArray>
['HFO', 'Diesel']
Length: 2, dtype: str


In [9]:
existingROUTE = df.loc[:,'route_id'].unique()
print (existingROUTE)

<StringArray>
['Warri-Bonny', 'Port Harcourt-Lagos', 'Lagos-Apapa', 'Escravos-Lagos']
Length: 4, dtype: str


In [10]:
existingWHETHERtype  = df.loc [:,'weather_conditions'].unique()
print (existingWHETHERtype)


<StringArray>
['Stormy', 'Moderate', 'Calm']
Length: 3, dtype: str


In [20]:
# to show the more risky trips according to the whether consition we check the value weather_conditions ='stormy'
# the engine efficiency indicator is fixed randomly ( it's not a real value of risk indicator)
riskyTrip = df [(df['weather_conditions'] == 'stormy') | (df['engine_efficiency'] < 80 ) ]
print(f"Total risky trips found: {len(riskyTrip)}")
riskyTrip.head ()



Total risky trips found: 562


,ship_id,ship_type,route_id,month,distance,fuel_type,fuel_consumption,CO2_emissions,weather_conditions,engine_efficiency
5,NG001,Oil Service Boat,Port Harcourt-Lagos,June,85.93,Diesel,2342.13,6753.42,Stormy,72.82
8,NG001,Oil Service Boat,Escravos-Lagos,September,116.44,Diesel,3661.75,9423.97,Calm,73.41
10,NG001,Oil Service Boat,Port Harcourt-Lagos,November,149.90,HFO,4014.14,11930.11,Stormy,76.53
14,NG002,Fishing Trawler,Escravos-Lagos,March,51.65,Diesel,1353.75,3649.81,Calm,74.32
16,NG002,Fishing Trawler,Port Harcourt-Lagos,May,72.25,Diesel,1873.13,5427.90,Stormy,78.01


In [12]:

# we want to create a flow map here for routes 
# we're doing it step by step 
import numpy as np
locations = [r.split('-') for r in existingROUTE ]
location = []
for start,destination in locations:
    if start not in location :location.append(start) 
    if destination not in location: location.append(destination)
location

# first find the gps cordination
cordination = []

from geopy.geocoders import Nominatim 
# we will use geopy for this idea 
import time 
loc = Nominatim (user_agent = "GetLoc")
for local in location:
    getLoc = loc.geocode(local+ ", Nigeria" ) 
    cordination.append([local,getLoc.latitude,getLoc.longitude])
    time.sleep(1)   
cordination 


[['Warri', 5.5186186, 5.7480196],
 ['Bonny', 4.505781, 7.203466],
 ['Port Harcourt', 4.7676576, 7.0188527],
 ['Lagos', 6.4550575, 3.3941795],
 ['Apapa', 6.445187, 3.3683732],
 ['Escravos', 5.6245844, 5.4861352]]

In [13]:
import plotly.graph_objects as go

# 1. Transform your cordination list into a quick-lookup dictionary
# This makes it easy to get lat/lon by city name
gps_dict = {item[0]: (item[1], item[2]) for item in cordination}

# 2. Initialize the Figure
fig = go.Figure()

# 3. Add the lines for each route
for start, destination in locations:
    if start in gps_dict and destination in gps_dict:
        start_lat, start_lon = gps_dict[start]
        end_lat, end_lon = gps_dict[destination]
        
        # Draw the line
        fig.add_trace(go.Scattergeo(
            lat=[start_lat, end_lat],
            lon=[start_lon, end_lon],
            mode='lines+markers',
            line=dict(width=2, color='blue'), # Removed 'shape' to fix the ValueError
            marker=dict(size=4, color='red'),
            name=f"{start} to {destination}"
        ))

# 4. Final Map Styling
fig.update_layout(
    title_text='<b>Maritime Flow Map: Ship Fuel Efficiency Study</b>',
    showlegend=False,
    geo=dict(
        scope='africa',
        showland=True,
        landcolor='rgb(243, 243, 243)',
        countrycolor='rgb(204, 204, 204)',
        # Zooming in specifically on the Bight of Benin / Nigeria coast
        center=dict(lat=5.5, lon=5.5),
        projection_scale=18 
    ),
    height=700
)
# get a little help to create it from youtube tuto ...
fig.show()

In [14]:
import plotly.express as px

# Relation Consommation, CO2 et Type de carburant
fig_co2 = px.scatter(
    df, 
    x='fuel_consumption', 
    y='CO2_emissions', 
    color='fuel_type',
    title="<b>Relation : Consommation, CO2 et Type de Carburant</b>",
    labels={'fuel_consumption': 'Consommation', 'CO2_emissions': 'Émissions CO2'},
    template='plotly_white'
)
fig_co2.show()

In [15]:
# Relation Efficacité, Consommation et Distance
fig_eff = px.scatter(
    df, 
    x='engine_efficiency', 
    y='fuel_consumption', 
    size='distance', 
    color='ship_type',
    hover_name='ship_id',
    title="<b>Efficacité vs Consommation (La taille du point = Distance)</b>",
    template='plotly_white'
)
fig_eff.show()

In [16]:
# Record de distance par Navire
max_dist_ship = df.loc[df.groupby('ship_id')['distance'].idxmax()]
print("--- the longest trip by ship---")
print(max_dist_ship[['ship_id', 'distance', 'route_id', 'month']])

--- the longest trip by ship---
     ship_id  distance             route_id     month
10     NG001    149.90  Port Harcourt-Lagos  November
15     NG002    197.46          Lagos-Apapa     April
30     NG003    149.48       Escravos-Lagos      July
40     NG004    192.04       Escravos-Lagos       May
49     NG005    494.24       Escravos-Lagos  February
...      ...       ...                  ...       ...
1382   NG116    453.83          Lagos-Apapa     March
1393   NG117    465.49  Port Harcourt-Lagos  February
1415   NG118    481.67          Warri-Bonny  December
1421   NG119    469.28          Lagos-Apapa      June
1437   NG120    193.09  Port Harcourt-Lagos   October

[120 rows x 4 columns]


In [17]:
# Record de distance par Mois
max_dist_month = df.loc[df.groupby('month')['distance'].idxmax()]
print("\n---longest trip by month ---")
print(max_dist_month[['month', 'ship_id', 'distance', 'route_id']])


---longest trip by month ---
          month ship_id  distance             route_id
975       April   NG082    498.18          Lagos-Apapa
187      August   NG016    453.47          Lagos-Apapa
95     December   NG008    497.16          Lagos-Apapa
793    February   NG067    498.55          Lagos-Apapa
372     January   NG032    453.14  Port Harcourt-Lagos
438        July   NG037    495.50  Port Harcourt-Lagos
977        June   NG082    478.32          Lagos-Apapa
134       March   NG012    490.16          Warri-Bonny
280         May   NG024    485.87  Port Harcourt-Lagos
202    November   NG017    483.05          Warri-Bonny
1173    October   NG098    463.72  Port Harcourt-Lagos
1016  September   NG085    487.34  Port Harcourt-Lagos


In [18]:
# Mean and Median per ship type
ship_stats = df.groupby('ship_type')[['engine_efficiency', 'fuel_consumption']].agg(['mean', 'median'])

print("--- Statistics per Ship Type ---")
print(ship_stats)

# Mode: Finding the most frequent values in the dataset
efficiency_mode = df['engine_efficiency'].mode()[0]
fuel_mode = df['fuel_consumption'].mode()[0]

print(f"\nMost frequent Efficiency (Mode): {efficiency_mode}%")
print(f"Most frequent Fuel Consumption (Mode): {fuel_mode} tons")

--- Statistics per Ship Type ---
                 engine_efficiency         fuel_consumption           
                              mean  median             mean     median
ship_type                                                             
Fishing Trawler          82.138300  82.165      3229.170933   3298.425
Oil Service Boat         83.419559  83.950      2619.435490   2667.055
Surfer Boat              82.444660  82.520      1666.133673   1680.505
Tanker Ship              82.183015  82.095     10780.408676  10729.510

Most frequent Efficiency (Mode): 82.94%
Most frequent Fuel Consumption (Mode): 855.01 tons


In [19]:
# Fuel consumption volatility based on weather
weather_volatility = df.groupby('weather_conditions')['fuel_consumption'].std()

print("--- Fuel Consumption Volatility (Std Dev) by Weather ---")
print(weather_volatility)

# Simple comparison
diff = weather_volatility['Stormy'] - weather_volatility['Calm']
print(f"\nFuel consumption is {diff:.2f} points more unstable in Stormy weather compared to Calm weather.")

--- Fuel Consumption Volatility (Std Dev) by Weather ---
weather_conditions
Calm        5101.744729
Moderate    4830.602439
Stormy      4714.382460
Name: fuel_consumption, dtype: float64

Fuel consumption is -387.36 points more unstable in Stormy weather compared to Calm weather.
